# PCB Defect Detection with Distributed Training

Multiclass defect detection using PyTorch Faster R-CNN in Snowflake Notebooks with distributed GPU training.

In [ ]:
# Get active Snowflake session (container notebook)
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Verify GPU availability and container runtime
import torch
import torchvision
print(f"PyTorch: {torch.__version__}")
print(f"TorchVision: {torchvision.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU Count: {torch.cuda.device_count()}")

## 1. Environment Setup

### Install Required Packages

- `torch` - PyTorch deep learning framework
- `torchvision` - Computer vision models and utilities
- `opencv` - Image processing
- `matplotlib` - Visualization
- `Pillow` - Image handling

### Import Libraries

In [ ]:
# Core imports
import os
import sys
import time
import warnings
warnings.filterwarnings("ignore")

# Data processing
import pandas as pd
import numpy as np

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

# Computer Vision
import torchvision.transforms as transforms
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import FasterRCNN

# Visualization
import matplotlib.pyplot as plt

# Snowflake ML
from snowflake.ml.registry import Registry
from snowflake.ml.modeling.distributors.pytorch import PyTorchDistributor, PyTorchScalingConfig, WorkerResourceConfig
from snowflake.ml.data.sharded_data_connector import ShardedDataConnector

# Set query tag for tracking
session.query_tag = {
    "origin": "sf_sit-is", 
    "name": "pcb_defect_detection_distributed",
    "version": {"major": 1, "minor": 0},
    "attributes": {"is_quickstart": 1, "source": "notebook"}
}

In [ ]:
# The NVIDIA A10G is a professional GPU designed for data center workloads, such as AI inference, virtual desktops
#  (VDI), and professional graphics. It's based on the NVIDIA Ampere architecture and features 24 GB of GDDR6
#  memory, 80 RT Cores, and 320 third-generation Tensor Cores, delivering up to 250 TOPS of compute power for AI. 
# Key features include its 300W power envelope, single-slot form factor, and versatility in handling both 
# graphically intensive and AI-accelerated applications. 

# Get device info
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print("Number of GPU devices available:", num_gpus)
    
    for i in range(num_gpus):
        print("Device", i, ":", torch.cuda.get_device_name(i))
    
    #Set a default device
    torch.cuda.set_device(0)
else:
    print("CUDA is not available. Check your installation or GPU setup.")

In [ ]:
#The PCB_CV_DEEP_PCB_DATASET_STAGE contains the PCB images loaded in the /images subfolder and the labels loaded in the /labels subfolder
#session.sql("ls @PCB_CV_DEEP_PCB_DATASET_STAGE")

In [ ]:
# Verify data loaded from stored procedure
print("Training Data:")
session.sql("SELECT COUNT(*) as count, COUNT(DISTINCT FILENAME) as images FROM TRAINING_DATA").show()

print("\nTest Data:")
session.sql("SELECT COUNT(*) as count, COUNT(DISTINCT FILENAME) as images FROM TEST_DATA").show()

print("\nDefect Class Distribution (Training):")
session.sql("""
    SELECT CLASS, COUNT(*) as count 
    FROM TRAINING_DATA 
    GROUP BY CLASS 
    ORDER BY CLASS
""").show()

## 2. Data Exploration

### View Training Dataset

In [ ]:
session.table("training_data").limit(5).collect();

## 3. Distributed Model Training

This section demonstrates how to train a Faster R-CNN object detection model using Snowflake's distributed training capabilities. The training process leverages multiple GPU workers in parallel to accelerate model training on the PCB defect detection dataset.

### Step 1: Define a Training Function for Each Worker

We create a `train_func()` that encapsulates the complete training logic for a single worker. This function will:
- Initialize the distributed training environment and establish communication between workers
- Load and preprocess the training data from Snowflake tables
- Create a custom PyTorch Dataset that decodes base64-encoded images and prepares bounding box annotations
- Initialize the Faster R-CNN model with a custom classifier head for our specific number of defect classes
- Wrap the model in DistributedDataParallel (DDP) to enable gradient synchronization across workers
- Execute the training loop with forward pass, loss calculation, backpropagation, and optimizer updates
- Save the trained model weights to a Snowflake stage for later inference

Each worker executes this function independently on its assigned data shard, with PyTorch's distributed training framework coordinating gradient updates across all workers.

### Step 2: Execute the Training Function Using PyTorchDistributor

The `PyTorchDistributor` is Snowflake's orchestration layer that manages the distributed training job. It handles:
- Provisioning GPU compute resources from the specified compute pool
- Distributing the training function code to all workers
- Coordinating the initialization and synchronization of the distributed training process
- Monitoring worker health and handling failures
- Collecting results and logs from all workers

Key components of the distributed training configuration:

* **ShardedDataConnector**: Automatically partitions the training dataset into non-overlapping shards, with each worker receiving a unique subset of the data. This ensures:
  - No data duplication across workers (each image is processed by exactly one worker per epoch)
  - Balanced workload distribution for optimal GPU utilization
  - Efficient data loading directly from Snowflake tables without manual partitioning logic

* **PyTorchScalingConfig**: Defines the distributed training cluster topology and resource allocation:
  - `num_workers`: Number of parallel training processes (typically matches the number of available GPUs)
  - `num_cpus_per_worker`: CPU cores allocated to each worker for data preprocessing and loading
  - `num_gpus_per_worker`: GPUs assigned to each worker (usually 1 GPU per worker for optimal performance)
  - `memory_per_worker`: RAM allocated per worker for caching data and model states
  - These settings directly impact training speed, memory usage, and cost

In [ ]:
# ==============================================================================
# IMPORTS: Required libraries for distributed training
# ==============================================================================
import base64  # For decoding base64-encoded image data from Snowflake
import io  # For handling in-memory byte streams
import cv2  # OpenCV for advanced image processing (if needed)
import torch  # PyTorch deep learning framework
import numpy as np  # Numerical computing library
from torch.utils.data import Dataset, DataLoader, IterableDataset  # Data loading utilities
from PIL import Image  # Python Imaging Library for image manipulation
import torchvision  # Computer vision utilities and models
from torchvision.models.detection import fasterrcnn_resnet50_fpn  # Pretrained Faster R-CNN model
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor  # Classifier head for Faster R-CNN
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights  # Pretrained weights
import torch.distributed as dist  # PyTorch distributed training utilities
from snowflake.ml.modeling.distributors.pytorch import get_context  # Snowflake distributed context
from torch.nn.parallel import DistributedDataParallel as DDP  # Distributed Data Parallel wrapper
import tempfile  # For creating temporary files/directories
import cloudpickle as cp  # Enhanced pickling for Python objects
import time  # For timing measurements

# ==============================================================================
# DIAGNOSTIC UTILITIES: Low-overhead performance monitoring
# ==============================================================================
class TrainingDiagnostics:
    """
    Lightweight diagnostics collector for distributed training.
    Uses CUDA events for non-blocking GPU timing and epoch-level aggregation
    to minimize overhead while providing useful insights.
    """
    def __init__(self, rank, world_size, sample_every_n_batches=10):
        self.rank = rank
        self.world_size = world_size
        self.sample_every_n_batches = sample_every_n_batches
        
        # Epoch-level timing
        self.epoch_start_time = None
        self.epoch_times = []
        
        # Batch-level timing (sampled)
        self.batch_times = []
        self.fetch_times = [] # Time spent waiting for data
        self.data_load_times = []
        self.forward_times = []
        self.backward_times = []
        
        # GPU memory tracking
        self.peak_memory_per_epoch = []
        
        # Throughput tracking
        self.total_images_processed = 0
        self.total_batches_processed = 0
        
        # CUDA events for async timing (non-blocking)
        self.use_cuda = torch.cuda.is_available()
        if self.use_cuda:
            self.start_event = torch.cuda.Event(enable_timing=True)
            self.end_event = torch.cuda.Event(enable_timing=True)
            self.data_start = torch.cuda.Event(enable_timing=True)
            self.data_end = torch.cuda.Event(enable_timing=True)
            self.forward_end = torch.cuda.Event(enable_timing=True)
            self.backward_end = torch.cuda.Event(enable_timing=True)
    
    def start_epoch(self):
        """Mark the start of an epoch."""
        self.epoch_start_time = time.perf_counter()
        if self.use_cuda:
            torch.cuda.reset_peak_memory_stats()
    
    def end_epoch(self, epoch_num, running_loss, num_batches, batch_size):
        """Record epoch-level metrics."""
        epoch_time = time.perf_counter() - self.epoch_start_time
        self.epoch_times.append(epoch_time)
        
        images_this_epoch = num_batches * batch_size
        self.total_images_processed += images_this_epoch
        self.total_batches_processed += num_batches
        
        # Capture peak GPU memory (minimal overhead - single call per epoch)
        if self.use_cuda:
            peak_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
            self.peak_memory_per_epoch.append(peak_memory_mb)
        else:
            peak_memory_mb = 0
        
        # Calculate throughput
        throughput = images_this_epoch / epoch_time if epoch_time > 0 else 0
        avg_loss = running_loss / (num_batches * batch_size) if (num_batches * batch_size) > 0 else 0
        
        # Print epoch summary
        print(f"[Rank {self.rank}] Epoch {epoch_num} | "
              f"Time: {epoch_time:.2f}s | "
              f"Throughput: {throughput:.1f} img/s | "
              f"Peak GPU Mem: {peak_memory_mb:.0f}MB | "
              f"Loss: {avg_loss:.4f}")
        
        return epoch_time, throughput, peak_memory_mb
    
    def should_sample_batch(self, batch_idx):
        """Determine if this batch should be timed (sampled approach)."""
        return batch_idx % self.sample_every_n_batches == 0
    
    def record_fetch_time(self, fetch_time_seconds):
        """Record time spent waiting for data loader."""
        self.fetch_times.append(fetch_time_seconds * 1000) # Convert to ms
    
    def record_batch_start(self):
        """Record start of a sampled batch (CUDA events for non-blocking timing)."""
        if self.use_cuda:
            self.data_start.record()
    
    def record_data_loaded(self):
        """Record when data loading completes."""
        if self.use_cuda:
            self.data_end.record()
    
    def record_forward_complete(self):
        """Record when forward pass completes."""
        if self.use_cuda:
            self.forward_end.record()
    
    def record_batch_complete(self):
        """Record batch completion and calculate timings."""
        if self.use_cuda:
            self.backward_end.record()
            
            # Synchronize only on sampled batches to get actual times
            # This is the only sync point, minimizing overhead
            torch.cuda.synchronize()
            
            # Calculate elapsed times from CUDA events
            data_time = self.data_start.elapsed_time(self.data_end)  # ms
            forward_time = self.data_end.elapsed_time(self.forward_end)  # ms
            backward_time = self.forward_end.elapsed_time(self.backward_end)  # ms
            total_time = self.data_start.elapsed_time(self.backward_end)  # ms
            
            self.data_load_times.append(data_time)
            self.forward_times.append(forward_time)
            self.backward_times.append(backward_time)
            self.batch_times.append(total_time)
    
    def print_summary(self):
        """Print final training summary with performance insights."""
        if self.rank != 0:
            return  # Only print from rank 0
        
        print("\n" + "="*70)
        print("                    TRAINING PERFORMANCE SUMMARY")
        print("="*70)
        
        # Overall timing
        total_time = sum(self.epoch_times)
        print(f"\nTIMING OVERVIEW")
        print(f"   Total Training Time: {total_time:.2f}s ({total_time/60:.1f} min)")
        print(f"   Average Epoch Time: {np.mean(self.epoch_times):.2f}s")
        print(f"   Epoch Time Std Dev: {np.std(self.epoch_times):.2f}s")
        
        # Throughput
        avg_throughput = self.total_images_processed / total_time if total_time > 0 else 0
        print(f"\nTHROUGHPUT")
        print(f"   Total Images Processed: {self.total_images_processed:,}")
        print(f"   Average Throughput: {avg_throughput:.1f} images/sec")
        print(f"   Effective Throughput (all {self.world_size} workers): {avg_throughput * self.world_size:.1f} images/sec")
        
        # GPU Memory
        if self.peak_memory_per_epoch:
            print(f"\nGPU MEMORY")
            print(f"   Peak Memory Usage: {max(self.peak_memory_per_epoch):.0f} MB")
            print(f"   Average Peak per Epoch: {np.mean(self.peak_memory_per_epoch):.0f} MB")
        
        # Batch timing breakdown (from sampled batches)
        if self.batch_times:
            print(f"\n BATCH TIMING BREAKDOWN (sampled every {self.sample_every_n_batches} batches)")
            avg_batch = np.mean(self.batch_times)
            avg_fetch = np.mean(self.fetch_times) if self.fetch_times else 0
            avg_data_transfer = np.mean(self.data_load_times)
            avg_forward = np.mean(self.forward_times)
            avg_backward = np.mean(self.backward_times)
            
            # Total batch cost including fetch
            total_avg_cost = avg_batch + avg_fetch
            
            print(f"   Average Batch Process Time: {avg_batch:.1f}ms (GPU/Compute)")
            print(f"   Average Data Fetch Time:    {avg_fetch:.1f}ms (CPU/IO Wait)")
            print(f"   Total Time Per Batch:       {total_avg_cost:.1f}ms")
            print(f"   ")
            print(f"   breakdown:")
            print(f"   ├─ Data Fetch (Wait): {avg_fetch:.1f}ms ({100*avg_fetch/total_avg_cost:.1f}%)")
            print(f"   ├─ Host-to-GPU:       {avg_data_transfer:.1f}ms ({100*avg_data_transfer/total_avg_cost:.1f}%)")
            print(f"   ├─ Forward Pass:      {avg_forward:.1f}ms ({100*avg_forward/total_avg_cost:.1f}%)")
            print(f"   └─ Backward Pass:     {avg_backward:.1f}ms ({100*avg_backward/total_avg_cost:.1f}%)")
            
            # Bottleneck detection
            if avg_fetch > avg_batch:
                 print(f"\nBOTTLENECK: Data Fetching is dominant! ({avg_fetch:.1f}ms vs {avg_batch:.1f}ms)")
                 print(f"       This indicates the GPU is starving.")
                 print(f"       Consider: Increasing num_workers in DataLoader, using local cache, or optimizing image decoding.")
            elif avg_data_transfer > avg_forward + avg_backward:
                print(f"\nBOTTLENECK: Host-to-GPU transfer is slow!")
            elif avg_backward > 2 * avg_forward:
                print(f"\nNote: Backward pass is {avg_backward/avg_forward:.1f}x slower than forward")
        
        print("\n" + "="*70)

    def save_to_json(self, output_dir="/tmp/diagnostics"):
        """Save diagnostics data to JSON file for post-training visualization."""
        import json
        import os
        
        os.makedirs(output_dir, exist_ok=True)
        
        # Compile all metrics into a dictionary
        total_time = sum(self.epoch_times) if self.epoch_times else 0
        avg_throughput = self.total_images_processed / total_time if total_time > 0 else 0
        
        data = {
            "rank": self.rank,
            "world_size": self.world_size,
            "epoch_times": self.epoch_times,
            "total_training_time": total_time,
            "total_images_processed": self.total_images_processed,
            "total_batches_processed": self.total_batches_processed,
            "avg_throughput_per_worker": avg_throughput,
            "peak_memory_per_epoch": self.peak_memory_per_epoch,
            "batch_times": self.batch_times,
            "fetch_times": self.fetch_times,
            "data_load_times": self.data_load_times,
            "forward_times": self.forward_times,
            "backward_times": self.backward_times,
            "sample_every_n_batches": self.sample_every_n_batches
        }
        
        filepath = os.path.join(output_dir, f"worker_{self.rank}.json")
        with open(filepath, "w") as f:
            json.dump(data, f, indent=2)
        
        print(f"[Rank {self.rank}] Diagnostics saved to {filepath}")

# ==============================================================================
# MAIN TRAINING FUNCTION: Executed by each worker in distributed training
# ==============================================================================
def train_func():
    # ------------------------------------------------------------------------------
    # MEMORY OPTIMIZATION: Set allocator config before CUDA init
    # ------------------------------------------------------------------------------
    import os
    # optimization to avoid fragmentation
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    # Suppress Ray SIGTERM warning in notebook environment
    os.environ["RAY_DISABLE_SIGTERM_HANDLER"] = "1"
    # ------------------------------------------------------------------------------
    # DISTRIBUTED TRAINING SETUP
    # ------------------------------------------------------------------------------
    # Get the distributed training context from Snowflake
    context = get_context()
    
    # Get this worker's rank (unique ID) in the distributed training cluster
    rank = context.get_rank()
    
    # Initialize the process group for distributed training using NCCL backend
    # NCCL (NVIDIA Collective Communications Library) is optimized for GPU communication
    dist.init_process_group(backend="nccl")
    
    # Print worker information for debugging
    print(f"Worker Rank : {rank}, world_size: {context.get_world_size()}")

    # ==============================================================================
    # CUSTOM DATASET CLASS: Transforms Snowflake data for PyTorch training
    # ==============================================================================
    
    # Class weights for oversampling minority classes (conservative to avoid OOM)
    # Using moderate oversampling to balance classes without excessive memory usage
    OVERSAMPLE_FACTOR = {
        1: 1,    # open - majority class, no oversampling
        2: 3,    # short - repeat 3x (48 -> 144)
        3: 2,    # mousebite - repeat 2x (103 -> 206)
        4: 5,    # spur - repeat 5x (10 -> 50)
        5: 5,    # copper - repeat 5x (8 -> 40)
        6: 5,    # pin-hole - repeat 5x (5 -> 25)
    }
    
    class FCBData(IterableDataset):
        """
        Custom PyTorch IterableDataset for PCB defect detection.
        Decodes base64 images from Snowflake and prepares targets for Faster R-CNN.
        
        Features:
        - Data augmentation (random flips, color jitter)
        - Oversampling of minority classes to address class imbalance
        """
        def __init__(self, source_dataset, transforms=None, augment=True):  
            # Store reference to the Snowflake dataset shard
            self.source_dataset = source_dataset
            self.augment = augment
            
            # Color jitter for augmentation
            self.color_jitter = torchvision.transforms.ColorJitter(
                brightness=0.2, contrast=0.2, saturation=0.1
            )
            
            # Set transform pipeline
            self.transforms = torchvision.transforms.Compose([
                torchvision.transforms.ToTensor(),
                torchvision.transforms.ConvertImageDtype(torch.float32),
            ])
        
        def apply_augmentation(self, image, boxes, img_width, img_height):
            """
            Apply random augmentations to image and adjust bounding boxes.
            """
            import random
            
            # Random horizontal flip (50% chance)
            if random.random() > 0.5:
                image = torchvision.transforms.functional.hflip(image)
                # Flip boxes: new_xmin = width - xmax, new_xmax = width - xmin
                boxes_flipped = boxes.clone()
                boxes_flipped[:, 0] = img_width - boxes[:, 2]  # new xmin
                boxes_flipped[:, 2] = img_width - boxes[:, 0]  # new xmax
                boxes = boxes_flipped
            
            # Random vertical flip (50% chance)
            if random.random() > 0.5:
                image = torchvision.transforms.functional.vflip(image)
                # Flip boxes: new_ymin = height - ymax, new_ymax = height - ymin
                boxes_flipped = boxes.clone()
                boxes_flipped[:, 1] = img_height - boxes[:, 3]  # new ymin
                boxes_flipped[:, 3] = img_height - boxes[:, 1]  # new ymax
                boxes = boxes_flipped
            
            return image, boxes
    
        def __iter__(self):
            """
            Iterator that yields (image, target) pairs for each data row.
            Oversamples minority classes based on OVERSAMPLE_FACTOR.
            """
            for row in self.source_dataset:
                # Get class label to determine oversampling factor
                class_label = int(row["CLASS"].item())
                oversample_count = OVERSAMPLE_FACTOR.get(class_label, 1)
                
                # Yield this sample multiple times for minority classes
                for _ in range(oversample_count):
                    # --------------------------------------------------------------
                    # IMAGE DECODING: Convert base64 string to PIL Image
                    # --------------------------------------------------------------
                    base64_image = row['IMAGE_DATA']
                    # Decode base64 string -> bytes -> PIL Image
                    pil_image = Image.open(io.BytesIO(base64.b64decode(base64_image)))
                    img_width, img_height = pil_image.size
                    
                    # Apply color jitter (before converting to tensor)
                    if self.augment:
                        pil_image = self.color_jitter(pil_image)
                    
                    # Apply transformations (converts PIL Image to tensor)
                    image = self.transforms(pil_image)
        
                    # --------------------------------------------------------------
                    # TARGET PREPARATION: Extract bounding boxes and class labels
                    # --------------------------------------------------------------
                    # Extract bounding box coordinates [xmin, ymin, xmax, ymax]
                    boxes = [[row[k].item() for k in ["XMIN", "YMIN", "XMAX", "YMAX"]] for _ in range(1)]
                    
                    # Extract class label (defect type: open, short, mousebite, etc.)
                    labels = [class_label]
        
                    # Convert to PyTorch tensors with appropriate dtypes
                    boxes = torch.as_tensor(boxes, dtype=torch.float32)  
                    labels = torch.as_tensor(labels, dtype=torch.int64)
                    
                    # Apply spatial augmentations (flips) - must adjust boxes too
                    if self.augment:
                        image, boxes = self.apply_augmentation(image, boxes, img_width, img_height)
                    
                    # --------------------------------------------------------------
                    # TARGET DICTIONARY: Format required by Faster R-CNN
                    # --------------------------------------------------------------
                    target = {  
                        'boxes': boxes,  # Bounding box coordinates [N, 4]
                        'labels': labels,  # Class labels [N]
                        'image_id': torch.tensor([int(row["FILENAME"])]),  # Unique image identifier
                        'area': (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]),  # Box area (width * height)
                        'iscrowd': torch.zeros((boxes.shape[0],), dtype=torch.uint8)  # 0 = single object, 1 = crowd
                    }
                    
                    # Yield (image, target) pair for this sample
                    yield (image, target)

    # ------------------------------------------------------------------------------
    # GPU CONTEXT: Set the GPU device for this worker
    # ------------------------------------------------------------------------------
    with torch.cuda.device(rank):
        # ==============================================================================
        # DATA LOADING: Prepare distributed dataset and dataloader
        # ==============================================================================
        # Get the dataset map from Snowflake's distributed context
        dataset_map = context.get_dataset_map()
        
        # Get this worker's shard of the training data
        # ShardedDataConnector automatically partitions data across workers
        train_shard = dataset_map["train"].get_shard().to_torch_dataset()
        
        # Wrap the shard with our custom dataset class
        train_dataset = FCBData(train_shard)
    
        # Get hyperparameters passed from the PyTorchDistributor
        hyper_parms = context.get_hyper_params()

        num_epochs = int(hyper_parms['num_epochs'])

        # ==============================================================================
        # MODEL INITIALIZATION: Load pretrained Faster R-CNN and customize for PCB defects
        # ==============================================================================
        # Load pretrained Faster R-CNN with ResNet50 backbone and Feature Pyramid Network
        weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT  
        model = fasterrcnn_resnet50_fpn(weights=weights)
          
        # ------------------------------------------------------------------------------
        # CLASSIFIER HEAD MODIFICATION: Adapt for custom number of defect classes
        # ------------------------------------------------------------------------------
        # Set number of classes (background + 6 defect types)
        num_classes = 7
        
        # Get the number of input features to the classification head
        in_features = model.roi_heads.box_predictor.cls_score.in_features  
        
        # Replace the pretrained predictor with a new one for our custom classes
        model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
        
        # Move model to the GPU assigned to this worker
        model.to(rank)
        
        # Wrap model with DistributedDataParallel for multi-GPU training
        # DDP synchronizes gradients across all workers during backpropagation
        model = DDP(model, device_ids=[rank])
        
         # ------------------------------------------------------------------------------
        # COLLATE FUNCTION: Custom batch collation for variable-sized inputs
        # ------------------------------------------------------------------------------
        def collate_fn(batch):
            """
            Custom collate function for Faster R-CNN.
            Object detection models accept lists of images and targets,
            not batched tensors (since images may have different sizes).
            """
            return tuple(zip(*batch))
    
        # Extract batch size from hyperparameters
        batch_size = int(hyper_parms['batch_size'])
        
        # Create DataLoader for efficient batch loading
        train_data_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=False,  # Shuffling handled by ShardedDataConnector
            collate_fn=collate_fn,  # Use custom collation
            pin_memory=True,  # Pin memory for faster GPU transfer
            pin_memory_device=f"cuda:{rank}",  # Pin to this worker's GPU
        )

        # ==============================================================================
        # OPTIMIZER AND SCHEDULER: Configure training optimization
        # ==============================================================================
        # Adam optimizer with weight decay for regularization
        # Only optimize parameters that require gradients
        optimizer = torch.optim.Adam(
            [p for p in model.parameters() if p.requires_grad], 
            lr=0.0001,  # Learning rate
            weight_decay=0.0005  # L2 regularization
        )

        
        # Learning rate scheduler: reduce LR by factor of 0.1 every 3 epochs
        # Helps model converge to better local minima
        lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

   

        

        # ==============================================================================
        # TRAINING LOOP: Train the model for specified number of epochs
        # ==============================================================================
        
        # Initialize diagnostics collector (samples every 10 batches for minimal overhead)
        diagnostics = TrainingDiagnostics(
            rank=rank, 
            world_size=context.get_world_size(),
            sample_every_n_batches=10
        )
        
        for epoch in range(num_epochs):
            # Set model to training mode (enables dropout, batch norm training, etc.)
            model.train()
            
            # Initialize metrics for this epoch
            running_loss = 0.0
            running_batches = 0
            
            # Start epoch timing
            diagnostics.start_epoch()
            
            # Track end of previous batch to measure fetch time
            batch_end_time = time.perf_counter()
            
            # ------------------------------------------------------------------------------
            # BATCH ITERATION: Process each batch of images and targets
            # ------------------------------------------------------------------------------
            for batch_idx, (images, targets) in enumerate(train_data_loader):
                # Calculate time spent waiting for data
                current_time = time.perf_counter()
                fetch_time = current_time - batch_end_time
                
                running_batches = running_batches + 1
                
                # Sampled batch timing (only every N batches to minimize overhead)
                should_time = diagnostics.should_sample_batch(batch_idx)
                
                if should_time:
                    diagnostics.record_fetch_time(fetch_time)
                    diagnostics.record_batch_start()
                
                # Transfer images to GPU
                images = [image.to(rank) for image in images]
                
                # Transfer all target tensors to GPU
                targets = [{k: v.to(rank) for k, v in t.items()} for t in targets]
                
                if should_time:
                    diagnostics.record_data_loaded()
                
                # ----------------------------------------------------------------------
                # FORWARD PASS: Compute losses
                # ----------------------------------------------------------------------
                # In training mode, Faster R-CNN returns a dictionary of losses
                # (classification loss, box regression loss, RPN losses, etc.)
                loss_dict = model(images, targets)
                
                # Sum all individual losses into a single scalar
                losses = sum(loss for loss in loss_dict.values())
                
                if should_time:
                    diagnostics.record_forward_complete()
                
                # ----------------------------------------------------------------------
                # BACKWARD PASS: Compute gradients and update weights
                # ----------------------------------------------------------------------
                # Clear gradients from previous iteration
                optimizer.zero_grad()
                
                # Compute gradients via backpropagation
                losses.backward()
                
                # Update model parameters using computed gradients
                optimizer.step()
                
                if should_time:
                    diagnostics.record_batch_complete()
    
                # Accumulate loss for epoch statistics
                running_loss += losses.item()
                
                # Update timestamp for next iteration's fetch time calculation
                batch_end_time = time.perf_counter()
    
            # Record epoch completion with diagnostics
            diagnostics.end_epoch(
                epoch_num=epoch + 1,
                running_loss=running_loss,
                num_batches=running_batches,
                batch_size=batch_size
            )
            
            # Step the learning rate scheduler (reduce LR if scheduled)
            lr_scheduler.step()
    
        # ==============================================================================
        # MODEL SAVING: Save trained model (only from rank 0 to avoid conflicts)
        # ==============================================================================
        MODEL_PATH = "/tmp/models/detectionmodel.pt"
        
        if rank == 0:
            # Only the primary worker (rank 0) saves the model
            # This prevents multiple workers from writing to the same file
            with open(MODEL_PATH, mode="w+b") as model_file:
                # Save model state dict (weights and biases)
                # Use model.module to access the underlying model inside DDP wrapper
                torch.save(model.module.state_dict(), model_file)
            print(f"Model written to {MODEL_PATH}")
    
        # Print performance diagnostics summary
        diagnostics.print_summary()

        # Save diagnostics to JSON for visualization
        diagnostics.save_to_json()

        # Training completion message
        print(f"[Rank {rank}] Training completed.")



### Training Configuration

1. Split the dataset (shard) for distributed training across multiple workers
2. Train using 4 workers, each with 1 GPU
3. Hyperparameters: `batch_size=16`, `epochs=10`

**Note:** Training with smaller batch size (16) and more epochs (10) helps the model learn better, especially with imbalanced class distributions in the DeepPCB dataset.

In [ ]:
import os
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # make the error point to the offending op

# Set up PyTorchDistributor
from snowflake.ml.modeling.distributors.pytorch import PyTorchDistributor, PyTorchScalingConfig, WorkerResourceConfig  
from snowflake.ml.data.sharded_data_connector import ShardedDataConnector  
import ray, torch

if not ray.is_initialized(): 
    ray.init()
print(ray.cluster_resources())        # should include 'GPU': N
print(torch.cuda.is_available(), torch.cuda.device_count())

df = session.table("training_data")

# Get total sample count for steps_per_epoch calculation
total_samples = df.count()
print(f"Total training samples: {total_samples}")

# Create sharded data connector.
train_data = ShardedDataConnector.from_dataframe(df)

# Create pytorch distributor.
pytorch_trainer = PyTorchDistributor(  
    train_func=train_func,
    scaling_config=PyTorchScalingConfig(  
        num_nodes=1,  
        num_workers_per_node=4,  
        resource_requirements_per_worker=WorkerResourceConfig(num_cpus=0, num_gpus=1),  
    )  
)  

# Run the trainer.
pytorch_trainer.run(
    dataset_map={"train": train_data},
    hyper_params={"batch_size": "16", "num_epochs": "10", "total_samples": str(total_samples)}
)

## Training Performance Visualization

Load worker diagnostics and create visualizations.

In [ ]:
# ==============================================================================
# TRAINING PERFORMANCE VISUALIZATION
# ==============================================================================
import json
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# ------------------------------------------------------------------------------
# Load diagnostics from all workers
# ------------------------------------------------------------------------------
diagnostics_dir = "/tmp/diagnostics"
worker_files = sorted(glob.glob(os.path.join(diagnostics_dir, "worker_*.json")))

if not worker_files:
    print("No diagnostics files found. Run training first.")
else:
    workers_data = []
    for f in worker_files:
        with open(f, 'r') as fp:
            workers_data.append(json.load(fp))
    
    print(f"Loaded diagnostics from {len(workers_data)} workers")
    
    # --------------------------------------------------------------------------
    # Dark Mode Color Scheme
    # --------------------------------------------------------------------------
    COLORS = {
        'background': '#1a1a2e',
        'panel': '#16213e',
        'grid': '#2d2d44',
        'text': '#e8e8e8',
        'text_secondary': '#a0a0a0',
        'cyan': '#00d4ff',
        'coral': '#ff6b6b',
        'teal': '#4ecdc4',
        'gold': '#ffe66d',
        'purple': '#a855f7',
        'green': '#22c55e'
    }
    
    # Apply dark style
    plt.style.use('dark_background')
    plt.rcParams.update({
        'figure.facecolor': COLORS['background'],
        'axes.facecolor': COLORS['panel'],
        'axes.edgecolor': COLORS['grid'],
        'axes.labelcolor': COLORS['text'],
        'text.color': COLORS['text'],
        'xtick.color': COLORS['text'],
        'ytick.color': COLORS['text'],
        'grid.color': COLORS['grid'],
        'grid.alpha': 0.3,
        'font.size': 10,
        'axes.titlesize': 12,
        'axes.labelsize': 10,
        'legend.facecolor': COLORS['panel'],
        'legend.edgecolor': COLORS['grid']
    })
    
    # --------------------------------------------------------------------------
    # Create Figure with Subplots
    # --------------------------------------------------------------------------
    fig = plt.figure(figsize=(16, 12))
    fig.patch.set_facecolor(COLORS['background'])
    
    # Title
    fig.suptitle('Distributed Training Performance Dashboard', 
                 fontsize=16, fontweight='bold', color=COLORS['cyan'], y=0.98)
    
    # Create grid layout
    gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.3, 
                          left=0.06, right=0.94, top=0.92, bottom=0.06)
    
    # --------------------------------------------------------------------------
    # Plot 1: Epoch Times Per Worker (Line Chart)
    # --------------------------------------------------------------------------
    ax1 = fig.add_subplot(gs[0, 0:2])
    
    colors_list = [COLORS['cyan'], COLORS['coral'], COLORS['teal'], COLORS['gold'], 
                   COLORS['purple'], COLORS['green']]
    
    for i, wd in enumerate(workers_data):
        epochs = range(1, len(wd['epoch_times']) + 1)
        color = colors_list[i % len(colors_list)]
        ax1.plot(epochs, wd['epoch_times'], 'o-', label=f"Worker {wd['rank']}", 
                 color=color, linewidth=2, markersize=6)
    
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Time (seconds)')
    ax1.set_title('Epoch Duration by Worker', fontweight='bold')
    ax1.legend(loc='upper right', framealpha=0.8)
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(0.5, len(workers_data[0]['epoch_times']) + 0.5)
    
    # --------------------------------------------------------------------------
    # Plot 2: Summary Stats Panel
    # --------------------------------------------------------------------------
    ax2 = fig.add_subplot(gs[0, 2])
    ax2.axis('off')
    
    # Calculate aggregate stats
    total_images = sum(wd['total_images_processed'] for wd in workers_data)
    total_time = max(wd['total_training_time'] for wd in workers_data)
    aggregate_throughput = sum(wd['avg_throughput_per_worker'] for wd in workers_data)
    avg_peak_memory = np.mean([max(wd['peak_memory_per_epoch']) for wd in workers_data if wd['peak_memory_per_epoch']])
    
    stats_text = f"""
    TRAINING SUMMARY
    ─────────────────────
    
    Workers:        {len(workers_data)}
    Total Time:     {total_time:.1f}s ({total_time/60:.1f} min)
    
    Images Total:   {total_images:,}
    Throughput:     {aggregate_throughput:.1f} img/s
    
    Peak GPU Mem:   {avg_peak_memory:.0f} MB (avg)
    """
    
    ax2.text(0.1, 0.95, stats_text, transform=ax2.transAxes, fontsize=11,
             verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round,pad=0.5', facecolor=COLORS['panel'], 
                       edgecolor=COLORS['cyan'], linewidth=2))
    
    # --------------------------------------------------------------------------
    # Plot 3: Throughput by Worker (Bar Chart)
    # --------------------------------------------------------------------------
    ax3 = fig.add_subplot(gs[1, 0])
    
    ranks = [wd['rank'] for wd in workers_data]
    throughputs = [wd['avg_throughput_per_worker'] for wd in workers_data]
    
    bars = ax3.bar(ranks, throughputs, color=COLORS['teal'], edgecolor=COLORS['cyan'], linewidth=1.5)
    ax3.set_xlabel('Worker Rank')
    ax3.set_ylabel('Images/Second')
    ax3.set_title('Throughput by Worker', fontweight='bold')
    ax3.set_xticks(ranks)
    ax3.grid(True, axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar, val in zip(bars, throughputs):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                 f'{val:.1f}', ha='center', va='bottom', fontsize=9, color=COLORS['text'])
    
    # --------------------------------------------------------------------------
    # Plot 4: Peak Memory by Worker (Bar Chart)
    # --------------------------------------------------------------------------
    ax4 = fig.add_subplot(gs[1, 1])
    
    peak_memories = [max(wd['peak_memory_per_epoch']) if wd['peak_memory_per_epoch'] else 0 for wd in workers_data]
    
    bars = ax4.bar(ranks, peak_memories, color=COLORS['coral'], edgecolor=COLORS['gold'], linewidth=1.5)
    ax4.set_xlabel('Worker Rank')
    ax4.set_ylabel('Memory (MB)')
    ax4.set_title('Peak GPU Memory by Worker', fontweight='bold')
    ax4.set_xticks(ranks)
    ax4.grid(True, axis='y', alpha=0.3)
    
    for bar, val in zip(bars, peak_memories):
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, 
                 f'{val:.0f}', ha='center', va='bottom', fontsize=9, color=COLORS['text'])
    
    # --------------------------------------------------------------------------
    # Plot 5: Batch Timing Breakdown (Stacked Horizontal Bar)
    # --------------------------------------------------------------------------
    ax5 = fig.add_subplot(gs[1, 2])
    
    # Aggregate timing across all workers
    all_fetch = []
    all_transfer = []
    all_forward = []
    all_backward = []
    
    for wd in workers_data:
        if wd['fetch_times']:
            all_fetch.extend(wd['fetch_times'])
        if wd['data_load_times']:
            all_transfer.extend(wd['data_load_times'])
        if wd['forward_times']:
            all_forward.extend(wd['forward_times'])
        if wd['backward_times']:
            all_backward.extend(wd['backward_times'])
    
    avg_fetch = np.mean(all_fetch) if all_fetch else 0
    avg_transfer = np.mean(all_transfer) if all_transfer else 0
    avg_forward = np.mean(all_forward) if all_forward else 0
    avg_backward = np.mean(all_backward) if all_backward else 0
    
    categories = ['Backward', 'Forward', 'Host->GPU', 'Data Fetch']
    values = [avg_backward, avg_forward, avg_transfer, avg_fetch]
    bar_colors = [COLORS['purple'], COLORS['teal'], COLORS['gold'], COLORS['coral']]
    
    bars = ax5.barh(categories, values, color=bar_colors, edgecolor=COLORS['text'], linewidth=0.5)
    ax5.set_xlabel('Time (ms)')
    ax5.set_title('Avg Batch Timing Breakdown', fontweight='bold')
    ax5.grid(True, axis='x', alpha=0.3)
    
    for bar, val in zip(bars, values):
        ax5.text(val + 1, bar.get_y() + bar.get_height()/2, 
                 f'{val:.1f}ms', ha='left', va='center', fontsize=9, color=COLORS['text'])
    
    # --------------------------------------------------------------------------
    # Plot 6: Epoch Time Distribution (Box Plot)
    # --------------------------------------------------------------------------
    ax6 = fig.add_subplot(gs[2, 0:2])
    
    epoch_data = [wd['epoch_times'] for wd in workers_data]
    bp = ax6.boxplot(epoch_data, patch_artist=True, labels=[f"W{wd['rank']}" for wd in workers_data])
    
    for patch, color in zip(bp['boxes'], colors_list[:len(workers_data)]):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    for whisker in bp['whiskers']:
        whisker.set_color(COLORS['text'])
    for cap in bp['caps']:
        cap.set_color(COLORS['text'])
    for median in bp['medians']:
        median.set_color(COLORS['gold'])
        median.set_linewidth(2)
    
    ax6.set_xlabel('Worker')
    ax6.set_ylabel('Epoch Time (seconds)')
    ax6.set_title('Epoch Time Distribution by Worker', fontweight='bold')
    ax6.grid(True, axis='y', alpha=0.3)
    
    # --------------------------------------------------------------------------
    # Plot 7: Timing Pie Chart
    # --------------------------------------------------------------------------
    ax7 = fig.add_subplot(gs[2, 2])
    
    total_batch_time = avg_fetch + avg_transfer + avg_forward + avg_backward
    if total_batch_time > 0:
        sizes = [avg_fetch, avg_transfer, avg_forward, avg_backward]
        labels = ['Data Fetch', 'Host->GPU', 'Forward', 'Backward']
        explode = (0.05, 0, 0, 0)
        
        wedges, texts, autotexts = ax7.pie(sizes, explode=explode, labels=labels, autopct='%1.1f%%',
                                            colors=[COLORS['coral'], COLORS['gold'], COLORS['teal'], COLORS['purple']],
                                            textprops={'color': COLORS['text']},
                                            wedgeprops={'edgecolor': COLORS['background'], 'linewidth': 2})
        
        for autotext in autotexts:
            autotext.set_color(COLORS['background'])
            autotext.set_fontweight('bold')
        
        ax7.set_title('Batch Time Distribution', fontweight='bold')
    
    # --------------------------------------------------------------------------
    # Save to local file and upload to Snowflake stage
    # --------------------------------------------------------------------------
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    local_path = f"/tmp/training_performance_{timestamp}.png"
    
    plt.savefig(local_path, dpi=150, facecolor=COLORS['background'], 
                edgecolor='none', bbox_inches='tight')
    
    print(f"Plot saved locally to: {local_path}")
    
    # Upload to Snowflake stage
    try:
        session.file.put(local_path, "@PCB_CV_DEEP_PCB_DATASET_STAGE/diagnostics/", auto_compress=False, overwrite=True)
        print(f"Plot uploaded to: @PCB_CV_DEEP_PCB_DATASET_STAGE/diagnostics/training_performance_{timestamp}.png")
    except Exception as e:
        print(f"Could not upload to stage: {e}")
    
    plt.show()
    print("Visualization complete!")


## 4. Model Evaluation

Evaluate the trained model on the **full test set** to compute performance metrics.

### Primary Metrics

| Metric | Description |
|--------|-------------|
| **Overall Accuracy** | Correct predictions / Total predictions |
| **Per-Defect TP/FP/FN** | True positives, false positives, false negatives per defect type |
| **Precision & Recall** | Per-class detection performance |

In [ ]:
# ==============================================================================
# EVALUATION UTILITIES: Metrics for Object Detection
# ==============================================================================

import numpy as np
from collections import defaultdict
import torch
import torchvision.transforms as transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor, FasterRCNN_ResNet50_FPN_Weights
from PIL import Image
import io
import base64

# Class labels for PCB defects
CLASS_NAMES = {
    0: "background",
    1: "open",
    2: "short", 
    3: "mousebite",
    4: "spur",
    5: "copper",
    6: "pin-hole"
}

def load_model(model_path):
    """Load the trained Faster R-CNN model."""
    weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    model = fasterrcnn_resnet50_fpn(weights=weights)
    
    num_classes = 7  # Background + 6 classes
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    model.load_state_dict(torch.load(model_path), strict=False)
    model.eval()
    return model

def decode_and_transform_image(base64_image):
    """Decode base64 image and transform for model input."""
    image_data = base64.b64decode(base64_image)
    image = Image.open(io.BytesIO(image_data)).convert('RGB')
    
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.ConvertImageDtype(torch.float32),
    ])
    return transform(image)

def compute_iou(box1, box2):
    """Compute Intersection over Union (IoU) between two bounding boxes."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0

def evaluate_predictions(predictions, ground_truths, iou_threshold=0.5, score_threshold=0.3):
    """
    Evaluate object detection predictions against ground truth.
    Returns precision, recall, F1, and per-class TP/FP/FN.
    
    Note: "Accuracy" here means detection accuracy (TP / all ground truths),
    which is actually RECALL in standard ML terminology.
    """
    # Per-class statistics
    class_stats = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})
    total_correct = 0
    total_predictions = 0
    total_ground_truths = 0
    
    for pred, gt in zip(predictions, ground_truths):
        pred_boxes = pred.get('boxes', [])
        pred_labels = pred.get('labels', [])
        pred_scores = pred.get('scores', [])
        
        gt_boxes = gt.get('boxes', [])
        gt_labels = gt.get('labels', [])
        
        # Filter predictions by score threshold (exclude background)
        valid_preds = [(b, l, s) for b, l, s in zip(pred_boxes, pred_labels, pred_scores) 
                       if s >= score_threshold and l != 0]
        valid_preds.sort(key=lambda x: x[2], reverse=True)
        
        # Track matched ground truths
        gt_matched = [False] * len(gt_boxes)
        
        for pred_box, pred_label, pred_score in valid_preds:
            total_predictions += 1
            best_iou = 0
            best_gt_idx = -1
            
            # Find best matching ground truth
            for gt_idx, (gt_box, gt_label) in enumerate(zip(gt_boxes, gt_labels)):
                if gt_matched[gt_idx] or gt_label != pred_label:
                    continue
                iou = compute_iou(pred_box, gt_box)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = gt_idx
            
            # True Positive or False Positive?
            if best_iou >= iou_threshold and best_gt_idx >= 0:
                class_stats[pred_label]['tp'] += 1
                gt_matched[best_gt_idx] = True
                total_correct += 1
            else:
                class_stats[pred_label]['fp'] += 1
        
        # Count False Negatives (unmatched ground truths)
        for gt_idx, (gt_label, matched) in enumerate(zip(gt_labels, gt_matched)):
            total_ground_truths += 1
            if not matched:
                class_stats[gt_label]['fn'] += 1
    
    # Compute overall metrics
    overall_recall = total_correct / total_ground_truths if total_ground_truths > 0 else 0
    overall_precision = total_correct / total_predictions if total_predictions > 0 else 0
    overall_f1 = (2 * overall_precision * overall_recall / (overall_precision + overall_recall) 
                  if (overall_precision + overall_recall) > 0 else 0)
    
    # Compile results
    results = {
        'overall_recall': round(overall_recall, 4),
        'overall_precision': round(overall_precision, 4),
        'overall_f1': round(overall_f1, 4),
        'total_correct': total_correct,
        'total_ground_truths': total_ground_truths,
        'total_predictions': total_predictions,
        'per_class': {}
    }
    
    # Per-class metrics
    for class_id in range(1, 7):  # Skip background (0)
        stats = class_stats[class_id]
        tp, fp, fn = stats['tp'], stats['fp'], stats['fn']
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        results['per_class'][CLASS_NAMES[class_id]] = {
            'precision': round(precision, 4),
            'recall': round(recall, 4),
            'f1': round(f1, 4),
            'TP': tp,
            'FP': fp,
            'FN': fn
        }
    
    return results

print("Evaluation utilities loaded!")

In [ ]:
# ==============================================================================
# TEST SET EVALUATION: Full test set evaluation
# ==============================================================================

import json
import base64
from PIL import Image
import io

# Load the trained model
model_path = '/tmp/models/detectionmodel.pt'
eval_model = load_model(model_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
eval_model = eval_model.to(device)
eval_model.eval()

# Load FULL test data
print("Loading test dataset...")
test_df = session.table("TEST_DATA").to_pandas()
print(f"   Found {len(test_df)} test samples")

# Run inference on full test set
predictions = []
ground_truths = []

print("\n Running inference on FULL test set...")
import time
inference_start_time = time.perf_counter()

with torch.no_grad():
    for idx, row in test_df.iterrows():
        image_tensor = decode_and_transform_image(row['IMAGE_DATA'])
        image_tensor = image_tensor.unsqueeze(0).to(device)
        
        output = eval_model(image_tensor)[0]
        
        predictions.append({
            'boxes': output['boxes'].cpu().numpy().tolist(),
            'labels': output['labels'].cpu().numpy().tolist(),
            'scores': output['scores'].cpu().numpy().tolist()
        })
        
        ground_truths.append({
            'boxes': [[row['XMIN'], row['YMIN'], row['XMAX'], row['YMAX']]],
            'labels': [int(row['CLASS'])]
        })
        
        if (idx + 1) % 100 == 0:
            elapsed = time.perf_counter() - inference_start_time
            avg_per_image = elapsed / (idx + 1) * 1000  # ms
            print(f"   Processed {idx + 1}/{len(test_df)} images... ({avg_per_image:.1f} ms/image)")

inference_end_time = time.perf_counter()
total_inference_time = inference_end_time - inference_start_time
avg_inference_time_ms = (total_inference_time / len(test_df)) * 1000

print(f"   Completed inference on {len(test_df)} images")
print(f"   Total inference time: {total_inference_time:.2f}s")
print(f"   Average time per image: {avg_inference_time_ms:.2f} ms")
print(f"   Throughput: {len(test_df) / total_inference_time:.1f} images/sec")

# Compute evaluation metrics
print("\n Computing metrics...")
eval_results = evaluate_predictions(predictions, ground_truths, iou_threshold=0.5, score_threshold=0.3)

# ==============================================================================
# DISPLAY RESULTS
# ==============================================================================
print("\n" + "="*70)
print("                      MODEL EVALUATION RESULTS")
print("="*70)

# PRIMARY METRIC: Overall Accuracy
print("\n" + "OVERALL ACCURACY")
print("-"*70)
print(f"   Accuracy: {eval_results['overall_recall']:.1%}")
print(f"   Correct Detections: {eval_results['total_correct']} / {eval_results['total_ground_truths']}")
print(f"   Total Predictions Made: {eval_results['total_predictions']}")

# PER-DEFECT DIAGNOSTICS
print("\n" + "PER-DEFECT DIAGNOSTICS (TP / FP / FN)")
print("-"*70)
print(f"{'Defect Type':<12} {'Accuracy':<10} {'TP':<8} {'FP':<8} {'FN':<8} {'Precision':<10} {'Recall':<10}")
print("-"*70)

for defect, metrics in eval_results['per_class'].items():
    print(f"{defect:<12} {metrics['recall']:<10.1%} {metrics['TP']:<8} {metrics['FP']:<8} {metrics['FN']:<8} {metrics['precision']:<10.2%} {metrics['recall']:<10.2%}")

print("-"*70)

# Summary
total_tp = sum(m['TP'] for m in eval_results['per_class'].values())
total_fp = sum(m['FP'] for m in eval_results['per_class'].values())
total_fn = sum(m['FN'] for m in eval_results['per_class'].values())
print(f"{'TOTAL':<12} {'':<10} {total_tp:<8} {total_fp:<8} {total_fn:<8}")
print("\n" + "="*70)

In [ ]:
# ==============================================================================
# VISUALIZATION: Per-Defect Performance Chart (Dark Mode)
# ==============================================================================

import matplotlib.pyplot as plt
import numpy as np

# Apply dark mode style
plt.style.use('dark_background')

# Define dark mode color palette (GitHub-inspired)
COLORS = {
    'background': '#0d1117',
    'surface': '#161b22',
    'border': '#30363d',
    'text_primary': '#e6edf3',
    'text_secondary': '#8b949e',
    'accent_green': '#3fb950',
    'accent_red': '#f85149',
    'accent_orange': '#d29922',
    'accent_blue': '#58a6ff',
    'accent_purple': '#a371f7',
    'accent_cyan': '#39d353',
}

# Prepare data
defect_names = list(eval_results['per_class'].keys())
tp_values = [eval_results['per_class'][d]['TP'] for d in defect_names]
fp_values = [eval_results['per_class'][d]['FP'] for d in defect_names]
fn_values = [eval_results['per_class'][d]['FN'] for d in defect_names]

# Create figure with dark background
fig, axes = plt.subplots(1, 2, figsize=(15, 6), facecolor=COLORS['background'])
fig.patch.set_facecolor(COLORS['background'])

# Main title with accent styling
fig.suptitle(f" MODEL DIAGNOSTICS", 
             fontsize=18, fontweight='bold', color=COLORS['accent_cyan'],
             y=0.98)
fig.text(0.5, 0.92, f"Overall Recall: {eval_results['overall_recall']:.1%}", 
         ha='center', fontsize=12, color=COLORS['text_secondary'])

# ===== LEFT CHART: Stacked Bar (TP/FP/FN) =====
ax1 = axes[0]
ax1.set_facecolor(COLORS['surface'])

x = np.arange(len(defect_names))
width = 0.65

# Create stacked bars with styled edges
bars_tp = ax1.bar(x, tp_values, width, label=' True Positives', 
                   color=COLORS['accent_green'], edgecolor=COLORS['border'], linewidth=0.5)
bars_fp = ax1.bar(x, fp_values, width, bottom=tp_values, 
                   label=' False Positives', color=COLORS['accent_red'], 
                   edgecolor=COLORS['border'], linewidth=0.5)
bars_fn = ax1.bar(x, fn_values, width, 
                   bottom=[tp+fp for tp,fp in zip(tp_values, fp_values)], 
                   label=' False Negatives', color=COLORS['accent_orange'],
                   edgecolor=COLORS['border'], linewidth=0.5)

ax1.set_xlabel('Defect Type', fontsize=11, color=COLORS['text_primary'], fontweight='medium')
ax1.set_ylabel('Count', fontsize=11, color=COLORS['text_primary'], fontweight='medium')
ax1.set_title('Detection Breakdown by Defect', fontsize=13, color=COLORS['text_primary'], 
              fontweight='bold', pad=15)
ax1.set_xticks(x)
ax1.set_xticklabels(defect_names, rotation=45, ha='right', fontsize=10, color=COLORS['text_secondary'])
ax1.tick_params(axis='y', colors=COLORS['text_secondary'])

# Styled legend
legend1 = ax1.legend(loc='upper right', facecolor=COLORS['surface'], 
                      edgecolor=COLORS['border'], fontsize=9)
for text in legend1.get_texts():
    text.set_color(COLORS['text_primary'])

# Subtle grid
ax1.grid(axis='y', alpha=0.15, color=COLORS['text_secondary'], linestyle='--')
ax1.set_axisbelow(True)

# Style spines
for spine in ax1.spines.values():
    spine.set_color(COLORS['border'])
    spine.set_linewidth(0.5)

# ===== RIGHT CHART: Horizontal Bar (Recall per Defect) =====
ax2 = axes[1]
ax2.set_facecolor(COLORS['surface'])

accuracies = [eval_results['per_class'][d]['recall'] for d in defect_names]

# Color bars based on performance threshold
colors = [COLORS['accent_blue'] if a >= 0.5 else COLORS['accent_red'] for a in accuracies]
bars = ax2.barh(defect_names, accuracies, color=colors, alpha=0.9, 
                 edgecolor=COLORS['border'], linewidth=0.5, height=0.7)

# Add overall recall reference line
ax2.axvline(x=eval_results['overall_recall'], color=COLORS['accent_purple'], 
            linestyle='--', linewidth=2.5, alpha=0.8,
            label=f"Overall: {eval_results['overall_recall']:.1%}")

ax2.set_xlabel('Recall', fontsize=11, color=COLORS['text_primary'], fontweight='medium')
ax2.set_title('Recall Performance by Defect', fontsize=13, color=COLORS['text_primary'], 
              fontweight='bold', pad=15)
ax2.set_xlim(0, 1.15)
ax2.tick_params(axis='x', colors=COLORS['text_secondary'])
ax2.tick_params(axis='y', colors=COLORS['text_secondary'])

# Styled legend
legend2 = ax2.legend(loc='lower right', facecolor=COLORS['surface'], 
                      edgecolor=COLORS['border'], fontsize=10)
for text in legend2.get_texts():
    text.set_color(COLORS['text_primary'])

# Add value labels with color-coded badges
for bar, acc in zip(bars, accuracies):
    badge_color = COLORS['accent_green'] if acc >= 0.7 else (COLORS['accent_orange'] if acc >= 0.5 else COLORS['accent_red'])
    ax2.text(acc + 0.03, bar.get_y() + bar.get_height()/2, f'{acc:.0%}', 
             va='center', fontsize=10, fontweight='bold', color=badge_color)

# Subtle grid
ax2.grid(axis='x', alpha=0.15, color=COLORS['text_secondary'], linestyle='--')
ax2.set_axisbelow(True)

# Style spines
for spine in ax2.spines.values():
    spine.set_color(COLORS['border'])
    spine.set_linewidth(0.5)

# Adjust layout
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.savefig('/tmp/model_diagnostics.png', dpi=150, bbox_inches='tight', 
            facecolor=COLORS['background'], edgecolor='none')
plt.show()

# Reset style to default for subsequent plots
plt.style.use('default')

print("\n Dark mode diagnostics chart saved to /tmp/model_diagnostics.png")

In [ ]:
# ==============================================================================
# EXTENDED DIAGNOSTICS: Comprehensive Model Performance Analysis (Dark Mode)
# ==============================================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.patches import Wedge
from matplotlib.collections import PatchCollection

# Apply dark mode style
plt.style.use('dark_background')

# Dark mode color palette
COLORS = {
    'background': '#0d1117',
    'surface': '#161b22',
    'surface_alt': '#1c2128',
    'border': '#30363d',
    'text_primary': '#e6edf3',
    'text_secondary': '#8b949e',
    'text_muted': '#6e7681',
    'accent_green': '#3fb950',
    'accent_red': '#f85149',
    'accent_orange': '#d29922',
    'accent_blue': '#58a6ff',
    'accent_purple': '#a371f7',
    'accent_cyan': '#39d353',
    'accent_pink': '#db61a2',
    'accent_yellow': '#e3b341',
}

# Extract data from eval_results
defect_names = list(eval_results['per_class'].keys())
precisions = [eval_results['per_class'][d]['precision'] for d in defect_names]
recalls = [eval_results['per_class'][d]['recall'] for d in defect_names]
f1_scores = [eval_results['per_class'][d]['f1'] for d in defect_names]
tp_values = [eval_results['per_class'][d]['TP'] for d in defect_names]
fp_values = [eval_results['per_class'][d]['FP'] for d in defect_names]
fn_values = [eval_results['per_class'][d]['FN'] for d in defect_names]

# ==============================================================================
# FIGURE 1: Precision-Recall Tradeoff & F1 Scores
# ==============================================================================
fig1, axes1 = plt.subplots(1, 3, figsize=(16, 5), facecolor=COLORS['background'])
fig1.patch.set_facecolor(COLORS['background'])
fig1.suptitle(" PRECISION-RECALL ANALYSIS", fontsize=16, fontweight='bold', 
              color=COLORS['accent_cyan'], y=1.02)

# --- Panel 1: Precision vs Recall Scatter ---
ax1 = axes1[0]
ax1.set_facecolor(COLORS['surface'])

# Create scatter with size based on sample count (TP + FN = ground truth)
sample_counts = [tp + fn for tp, fn in zip(tp_values, fn_values)]
sizes = [max(80, s * 3) for s in sample_counts]

# Color gradient based on F1 score
scatter_colors = [COLORS['accent_green'] if f >= 0.6 else 
                  COLORS['accent_orange'] if f >= 0.3 else 
                  COLORS['accent_red'] for f in f1_scores]

scatter = ax1.scatter(recalls, precisions, s=sizes, c=scatter_colors, 
                       alpha=0.85, edgecolors=COLORS['border'], linewidths=1.5)

# Add labels for each point
for i, name in enumerate(defect_names):
    ax1.annotate(name, (recalls[i], precisions[i]), 
                 textcoords="offset points", xytext=(8, 5),
                 fontsize=9, color=COLORS['text_primary'], fontweight='medium')

# Add iso-F1 curves
for f1_val in [0.2, 0.4, 0.6, 0.8]:
    r_range = np.linspace(0.01, 1.0, 100)
    p_range = (f1_val * r_range) / (2 * r_range - f1_val)
    valid = (p_range > 0) & (p_range <= 1)
    ax1.plot(r_range[valid], p_range[valid], '--', alpha=0.3, 
             color=COLORS['text_muted'], linewidth=1)
    # Label the curve
    if f1_val == 0.6:
        ax1.text(0.95, (f1_val * 0.95) / (2 * 0.95 - f1_val) + 0.02, 
                 f'F1={f1_val}', fontsize=8, color=COLORS['text_muted'])

ax1.set_xlabel('Recall', fontsize=11, color=COLORS['text_primary'])
ax1.set_ylabel('Precision', fontsize=11, color=COLORS['text_primary'])
ax1.set_title('Precision vs Recall\n(size = sample count)', fontsize=12, 
              color=COLORS['text_primary'], pad=10)
ax1.set_xlim(-0.05, 1.1)
ax1.set_ylim(-0.05, 1.1)
ax1.tick_params(colors=COLORS['text_secondary'])
ax1.grid(alpha=0.15, color=COLORS['text_secondary'], linestyle='--')
for spine in ax1.spines.values():
    spine.set_color(COLORS['border'])

# --- Panel 2: F1 Score Comparison ---
ax2 = axes1[1]
ax2.set_facecolor(COLORS['surface'])

y_pos = np.arange(len(defect_names))
f1_colors = [COLORS['accent_green'] if f >= 0.6 else 
             COLORS['accent_orange'] if f >= 0.3 else 
             COLORS['accent_red'] for f in f1_scores]

bars = ax2.barh(y_pos, f1_scores, color=f1_colors, alpha=0.9,
                 edgecolor=COLORS['border'], linewidth=0.5, height=0.7)

# Add value labels
for i, (bar, f1) in enumerate(zip(bars, f1_scores)):
    ax2.text(f1 + 0.02, bar.get_y() + bar.get_height()/2, f'{f1:.2f}', 
             va='center', fontsize=10, fontweight='bold', color=COLORS['text_primary'])

# Average F1 line
avg_f1 = np.mean(f1_scores)
ax2.axvline(x=avg_f1, color=COLORS['accent_purple'], linestyle='--', 
            linewidth=2, label=f'Avg: {avg_f1:.2f}')

ax2.set_yticks(y_pos)
ax2.set_yticklabels(defect_names, fontsize=10, color=COLORS['text_secondary'])
ax2.set_xlabel('F1 Score', fontsize=11, color=COLORS['text_primary'])
ax2.set_title('F1 Score by Defect Type', fontsize=12, color=COLORS['text_primary'], pad=10)
ax2.set_xlim(0, 1.15)
ax2.tick_params(axis='x', colors=COLORS['text_secondary'])
ax2.legend(facecolor=COLORS['surface'], edgecolor=COLORS['border'], 
           fontsize=9, labelcolor=COLORS['text_primary'])
ax2.grid(axis='x', alpha=0.15, color=COLORS['text_secondary'], linestyle='--')
for spine in ax2.spines.values():
    spine.set_color(COLORS['border'])

# --- Panel 3: Precision & Recall Side-by-Side ---
ax3 = axes1[2]
ax3.set_facecolor(COLORS['surface'])

x_pos = np.arange(len(defect_names))
width = 0.35

bars_p = ax3.bar(x_pos - width/2, precisions, width, label='Precision', 
                  color=COLORS['accent_blue'], edgecolor=COLORS['border'], alpha=0.9)
bars_r = ax3.bar(x_pos + width/2, recalls, width, label='Recall', 
                  color=COLORS['accent_cyan'], edgecolor=COLORS['border'], alpha=0.9)

ax3.set_xticks(x_pos)
ax3.set_xticklabels(defect_names, rotation=45, ha='right', fontsize=9, 
                    color=COLORS['text_secondary'])
ax3.set_ylabel('Score', fontsize=11, color=COLORS['text_primary'])
ax3.set_title('Precision vs Recall Comparison', fontsize=12, 
              color=COLORS['text_primary'], pad=10)
ax3.set_ylim(0, 1.1)
ax3.tick_params(axis='y', colors=COLORS['text_secondary'])
legend3 = ax3.legend(facecolor=COLORS['surface'], edgecolor=COLORS['border'], fontsize=9)
for text in legend3.get_texts():
    text.set_color(COLORS['text_primary'])
ax3.grid(axis='y', alpha=0.15, color=COLORS['text_secondary'], linestyle='--')
for spine in ax3.spines.values():
    spine.set_color(COLORS['border'])

plt.tight_layout()
plt.savefig('/tmp/precision_recall_analysis.png', dpi=150, bbox_inches='tight', 
            facecolor=COLORS['background'])
plt.show()

# ==============================================================================
# FIGURE 2: Error Analysis & Detection Volume
# ==============================================================================
fig2, axes2 = plt.subplots(1, 3, figsize=(16, 5), facecolor=COLORS['background'])
fig2.patch.set_facecolor(COLORS['background'])
fig2.suptitle(" ERROR ANALYSIS & DETECTION VOLUME", fontsize=16, fontweight='bold', 
              color=COLORS['accent_orange'], y=1.02)

# --- Panel 1: Error Rate Analysis (FP Rate vs FN Rate) ---
ax4 = axes2[0]
ax4.set_facecolor(COLORS['surface'])

# Calculate error rates
total_preds = [tp + fp for tp, fp in zip(tp_values, fp_values)]
total_gt = [tp + fn for tp, fn in zip(tp_values, fn_values)]
fp_rates = [fp / pred if pred > 0 else 0 for fp, pred in zip(fp_values, total_preds)]
fn_rates = [fn / gt if gt > 0 else 0 for fn, gt in zip(fn_values, total_gt)]

scatter2 = ax4.scatter(fp_rates, fn_rates, s=150, c=COLORS['accent_pink'], 
                        alpha=0.85, edgecolors=COLORS['border'], linewidths=1.5)

for i, name in enumerate(defect_names):
    ax4.annotate(name, (fp_rates[i], fn_rates[i]), 
                 textcoords="offset points", xytext=(8, 5),
                 fontsize=9, color=COLORS['text_primary'], fontweight='medium')

# Add quadrant reference
ax4.axhline(y=0.5, color=COLORS['text_muted'], linestyle=':', alpha=0.5)
ax4.axvline(x=0.5, color=COLORS['text_muted'], linestyle=':', alpha=0.5)

# Quadrant labels
ax4.text(0.75, 0.75, 'High Both\n(Needs Work)', ha='center', fontsize=8, 
         color=COLORS['accent_red'], alpha=0.7)
ax4.text(0.25, 0.25, 'Low Both\n(Good)', ha='center', fontsize=8, 
         color=COLORS['accent_green'], alpha=0.7)

ax4.set_xlabel('False Positive Rate\n(FP / Total Predictions)', fontsize=10, 
               color=COLORS['text_primary'])
ax4.set_ylabel('False Negative Rate\n(FN / Ground Truth)', fontsize=10, 
               color=COLORS['text_primary'])
ax4.set_title('Error Rate Analysis', fontsize=12, color=COLORS['text_primary'], pad=10)
ax4.set_xlim(-0.05, 1.05)
ax4.set_ylim(-0.05, 1.05)
ax4.tick_params(colors=COLORS['text_secondary'])
ax4.grid(alpha=0.15, color=COLORS['text_secondary'], linestyle='--')
for spine in ax4.spines.values():
    spine.set_color(COLORS['border'])

# --- Panel 2: Detection Volume (Predicted vs Ground Truth) ---
ax5 = axes2[1]
ax5.set_facecolor(COLORS['surface'])

x_pos = np.arange(len(defect_names))
width = 0.35

# Predictions made (TP + FP) vs Ground truth (TP + FN)
predictions_made = [tp + fp for tp, fp in zip(tp_values, fp_values)]
ground_truth_count = [tp + fn for tp, fn in zip(tp_values, fn_values)]

bars_gt = ax5.bar(x_pos - width/2, ground_truth_count, width, label='Ground Truth', 
                   color=COLORS['accent_purple'], edgecolor=COLORS['border'], alpha=0.9)
bars_pred = ax5.bar(x_pos + width/2, predictions_made, width, label='Predictions', 
                     color=COLORS['accent_yellow'], edgecolor=COLORS['border'], alpha=0.9)

ax5.set_xticks(x_pos)
ax5.set_xticklabels(defect_names, rotation=45, ha='right', fontsize=9, 
                    color=COLORS['text_secondary'])
ax5.set_ylabel('Count', fontsize=11, color=COLORS['text_primary'])
ax5.set_title('Detection Volume:\nGround Truth vs Predictions', fontsize=12, 
              color=COLORS['text_primary'], pad=10)
ax5.tick_params(axis='y', colors=COLORS['text_secondary'])
legend5 = ax5.legend(facecolor=COLORS['surface'], edgecolor=COLORS['border'], fontsize=9)
for text in legend5.get_texts():
    text.set_color(COLORS['text_primary'])
ax5.grid(axis='y', alpha=0.15, color=COLORS['text_secondary'], linestyle='--')
for spine in ax5.spines.values():
    spine.set_color(COLORS['border'])

# --- Panel 3: Correct vs Errors per Class (Normalized) ---
ax6 = axes2[2]
ax6.set_facecolor(COLORS['surface'])

# Normalize: TP / (TP + FP + FN) for each class
total_instances = [tp + fp + fn for tp, fp, fn in zip(tp_values, fp_values, fn_values)]
tp_norm = [tp / t if t > 0 else 0 for tp, t in zip(tp_values, total_instances)]
fp_norm = [fp / t if t > 0 else 0 for fp, t in zip(fp_values, total_instances)]
fn_norm = [fn / t if t > 0 else 0 for fn, t in zip(fn_values, total_instances)]

x_pos = np.arange(len(defect_names))
width = 0.7

ax6.bar(x_pos, tp_norm, width, label='Correct (TP)', color=COLORS['accent_green'], 
        edgecolor=COLORS['border'])
ax6.bar(x_pos, fp_norm, width, bottom=tp_norm, label='False Alarm (FP)', 
        color=COLORS['accent_red'], edgecolor=COLORS['border'])
ax6.bar(x_pos, fn_norm, width, bottom=[t + f for t, f in zip(tp_norm, fp_norm)], 
        label='Missed (FN)', color=COLORS['accent_orange'], edgecolor=COLORS['border'])

ax6.set_xticks(x_pos)
ax6.set_xticklabels(defect_names, rotation=45, ha='right', fontsize=9, 
                    color=COLORS['text_secondary'])
ax6.set_ylabel('Proportion', fontsize=11, color=COLORS['text_primary'])
ax6.set_title('Normalized Error Distribution', fontsize=12, 
              color=COLORS['text_primary'], pad=10)
ax6.set_ylim(0, 1.05)
ax6.tick_params(axis='y', colors=COLORS['text_secondary'])
legend6 = ax6.legend(loc='upper right', facecolor=COLORS['surface'], 
                      edgecolor=COLORS['border'], fontsize=9)
for text in legend6.get_texts():
    text.set_color(COLORS['text_primary'])
ax6.grid(axis='y', alpha=0.15, color=COLORS['text_secondary'], linestyle='--')
for spine in ax6.spines.values():
    spine.set_color(COLORS['border'])

plt.tight_layout()
plt.savefig('/tmp/error_analysis.png', dpi=150, bbox_inches='tight', 
            facecolor=COLORS['background'])
plt.show()

# ==============================================================================
# FIGURE 3: Overall Model Summary & Radar Chart
# ==============================================================================
fig3, axes3 = plt.subplots(1, 3, figsize=(16, 5), facecolor=COLORS['background'])
fig3.patch.set_facecolor(COLORS['background'])
fig3.suptitle(" OVERALL MODEL SUMMARY", fontsize=16, fontweight='bold', 
              color=COLORS['accent_purple'], y=1.02)

# --- Panel 1: Overall Metrics Donut Chart ---
ax7 = axes3[0]
ax7.set_facecolor(COLORS['background'])

correct = eval_results['total_correct']
total_gt = eval_results['total_ground_truths']
missed = total_gt - correct
total_preds = eval_results['total_predictions']
false_alarms = total_preds - correct

# Create donut
sizes = [correct, missed, false_alarms]
labels = ['Correct\nDetections', 'Missed\n(FN)', 'False\nAlarms (FP)']
colors_donut = [COLORS['accent_green'], COLORS['accent_orange'], COLORS['accent_red']]
explode = (0.02, 0.02, 0.02)

wedges, texts, autotexts = ax7.pie(sizes, explode=explode, labels=labels, colors=colors_donut,
                                    autopct='%1.1f%%', startangle=90, pctdistance=0.75,
                                    wedgeprops=dict(width=0.5, edgecolor=COLORS['background']))

for text in texts:
    text.set_color(COLORS['text_primary'])
    text.set_fontsize(9)
for autotext in autotexts:
    autotext.set_color(COLORS['text_primary'])
    autotext.set_fontsize(10)
    autotext.set_fontweight('bold')

# Center text
ax7.text(0, 0, f"{eval_results['overall_recall']:.0%}\nRecall", ha='center', va='center',
         fontsize=16, fontweight='bold', color=COLORS['accent_cyan'])
ax7.set_title('Detection Outcome Distribution', fontsize=12, 
              color=COLORS['text_primary'], pad=15)

# --- Panel 2: Radar Chart of Per-Class F1 ---
ax8 = axes3[1]
ax8.remove()  # Remove rectangular axes
ax8 = fig3.add_subplot(1, 3, 2, projection='polar', facecolor=COLORS['surface'])

# Radar chart data
categories = defect_names
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]  # Complete the loop

f1_radar = f1_scores + f1_scores[:1]
recall_radar = recalls + recalls[:1]
precision_radar = precisions + precisions[:1]

ax8.set_facecolor(COLORS['surface'])
ax8.plot(angles, f1_radar, 'o-', linewidth=2, color=COLORS['accent_green'], 
         label='F1 Score', markersize=6)
ax8.fill(angles, f1_radar, alpha=0.25, color=COLORS['accent_green'])

ax8.plot(angles, recall_radar, 'o-', linewidth=2, color=COLORS['accent_blue'], 
         label='Recall', markersize=6)
ax8.fill(angles, recall_radar, alpha=0.15, color=COLORS['accent_blue'])

ax8.plot(angles, precision_radar, 'o-', linewidth=2, color=COLORS['accent_purple'], 
         label='Precision', markersize=6)
ax8.fill(angles, precision_radar, alpha=0.15, color=COLORS['accent_purple'])

ax8.set_xticks(angles[:-1])
ax8.set_xticklabels(categories, fontsize=9, color=COLORS['text_primary'])
ax8.set_ylim(0, 1)
ax8.tick_params(colors=COLORS['text_secondary'])
ax8.grid(color=COLORS['border'], alpha=0.5)
ax8.spines['polar'].set_color(COLORS['border'])

legend8 = ax8.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), 
                      facecolor=COLORS['surface'], edgecolor=COLORS['border'], fontsize=9)
for text in legend8.get_texts():
    text.set_color(COLORS['text_primary'])
ax8.set_title('Per-Class Metrics Radar', fontsize=12, color=COLORS['text_primary'], pad=20)

# --- Panel 3: Summary Statistics Table ---
ax9 = axes3[2]
ax9.set_facecolor(COLORS['surface'])
ax9.axis('off')

# Create summary table
summary_data = [
    ['Metric', 'Value'],
    ['─' * 20, '─' * 10],
    ['Overall Recall', f"{eval_results['overall_recall']:.1%}"],
    ['Overall Precision', f"{eval_results['overall_precision']:.1%}"],
    ['Overall F1 Score', f"{eval_results['overall_f1']:.2f}"],
    ['─' * 20, '─' * 10],
    ['Correct Detections', f"{eval_results['total_correct']:,}"],
    ['Total Ground Truth', f"{eval_results['total_ground_truths']:,}"],
    ['Total Predictions', f"{eval_results['total_predictions']:,}"],
    ['─' * 20, '─' * 10],
    ['Total TP', f"{sum(tp_values):,}"],
    ['Total FP', f"{sum(fp_values):,}"],
    ['Total FN', f"{sum(fn_values):,}"],
    ['─' * 20, '─' * 10],
    ['Best F1 (class)', f"{defect_names[np.argmax(f1_scores)]} ({max(f1_scores):.2f})"],
    ['Worst F1 (class)', f"{defect_names[np.argmin(f1_scores)]} ({min(f1_scores):.2f})"],
]

y_start = 0.95
for i, (label, value) in enumerate(summary_data):
    y_pos = y_start - i * 0.058
    
    if '─' in label:
        ax9.text(0.1, y_pos, label + value, fontsize=9, color=COLORS['border'],
                 transform=ax9.transAxes, family='monospace')
    else:
        ax9.text(0.1, y_pos, label, fontsize=10, color=COLORS['text_secondary'],
                 transform=ax9.transAxes, fontweight='medium')
        ax9.text(0.7, y_pos, value, fontsize=10, color=COLORS['text_primary'],
                 transform=ax9.transAxes, fontweight='bold')

ax9.set_title('Model Summary Statistics', fontsize=12, color=COLORS['text_primary'], 
              pad=15, loc='left', x=0.1)

# Add border
rect = plt.Rectangle((0.05, 0.02), 0.9, 0.93, fill=False, 
                       edgecolor=COLORS['border'], linewidth=1, transform=ax9.transAxes)
ax9.add_patch(rect)

plt.tight_layout()
plt.savefig('/tmp/model_summary.png', dpi=150, bbox_inches='tight', 
            facecolor=COLORS['background'])
plt.show()

# Reset to default style
plt.style.use('default')

print("\n Extended diagnostics saved:")
print("   • /tmp/precision_recall_analysis.png")
print("   • /tmp/error_analysis.png")
print("   • /tmp/model_summary.png")


## 5. Model Deployment

Deploy the trained model to Snowflake for production inference.

### Snowflake Model Registry

The model registry stores ML models as first-class schema-level objects in Snowflake.

**Steps:**
1. Load the trained model from disk
2. Define a custom inference wrapper
3. Log to Model Registry with dependencies
4. Deploy as a GPU-accelerated service

In [ ]:
# import os
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # make the error point to the offending op

import torch
import torchvision.transforms as transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor, FasterRCNN_ResNet50_FPN_Weights
from PIL import Image
import io
import json
import base64

import importlib, pkg_resources as pr
print("torch", pr.get_distribution("torch").version)
print("torchvision", pr.get_distribution("torchvision").version)
print("transformers", pr.get_distribution("transformers").version)
print("accelerate", pr.get_distribution("accelerate").version)
print("PyTorch:", torch.__version__)
print("CUDA (built with):", torch.version.cuda)        # e.g., '12.1'
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name}, CC {props.major}.{props.minor}, "
          f"VRAM {props.total_memory / (1024**3):.1f} GB")

# Optional: cuDNN version (if present)
print("cuDNN:", torch.backends.cudnn.version())

print("snowflake.ml.python", pr.get_distribution("snowflake.ml.python").version)




df=session.table("training_data").limit(1).to_pandas()

first_row = df.iloc[0]  
base64_image = first_row['IMAGE_DATA'] 
df = pd.DataFrame({'IMAGE_DATA': [base64_image]})  

spdf=session.create_dataframe(df)
# Function to load the model
def load_model(model_path):  
    weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT  
    model = fasterrcnn_resnet50_fpn(weights=weights)  
    
    num_classes = 7  # Background + 6 classes
    in_features = model.roi_heads.box_predictor.cls_score.in_features  
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)  
    model.load_state_dict(torch.load(model_path), strict=False)  
    model.eval()  
    return model 

# Function to decode and transform an image
def decode_and_transform_image(base64_image):  
    image_data = base64.b64decode(base64_image)  
    image = Image.open(io.BytesIO(image_data)).convert('RGB')  
    
    # Use the SAME transforms as training
    transform = transforms.Compose([  
        transforms.ToTensor(),  # Converts to [C, H, W]
        transforms.ConvertImageDtype(torch.float32),  # Match training dtype
    ])  
    image_tensor = transform(image)
    
    print(f"Shape after transformation: {image_tensor.shape}")
    return image_tensor


# try:
model_path = '/tmp/models/detectionmodel.pt'
model = load_model(model_path)

from snowflake.ml.model import custom_model

class DefectDetectionModel(custom_model.CustomModel):
    def __init__(self, context: custom_model.ModelContext) -> None:
        super().__init__(context)

    @custom_model.inference_api
    def predict(self, input_df: pd.DataFrame) -> pd.DataFrame:
        import torch
        import json
    
        # Device inside the inference container
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
        # Move model to device
        model = self.context.model_ref("rcnn")
        model = model.to(device)
        model.eval()

        processed_input = torch.stack(
        input_df['IMAGE_DATA'].apply(decode_and_transform_image).to_list()).to(device)
        with torch.no_grad():
            raw_outputs = model(processed_input)

        # Back to CPU for JSON serialization
        final_output = pd.DataFrame({
            "output": [
                json.dumps({k: v.detach().cpu().numpy().tolist() for k, v in res.items()})
                for res in raw_outputs
            ]
        })
    
        return final_output

ddm = DefectDetectionModel(context = custom_model.ModelContext(models={'rcnn': model}))


ml_reg = Registry(session=session)  
# Drop dependent services before deleting the model
session.sql("DROP SERVICE IF EXISTS PCB_CV.PUBLIC.DEFECTDETECTSERVICE").collect()
# Drop any MODEL_BUILD_% services
services_df = session.sql("SHOW SERVICES IN SCHEMA PCB_CV.PUBLIC").collect()
for svc in services_df:
    svc_name = svc["name"]
    if svc_name.startswith("MODEL_BUILD_"):
        session.sql(f"DROP SERVICE IF EXISTS PCB_CV.PUBLIC.{svc_name}").collect()

# Delete the model and all its versions
try:
    ml_reg.delete_model("DefectDetectionModel")
except Exception as e:
    print(f"Could not delete model: {e}")

# Log the model with the sample input for Snowflake registry
mv = ml_reg.log_model(  
    ddm,  
    model_name="DefectDetectionModel",  
    version_name='v1',  
    pip_requirements=["torch==2.6.0", "torchvision==0.21.0"],
    sample_input_data=spdf,
    conda_dependencies=["pytorch", "torchvision"],
    options={"embed_local_ml_library": True, "relax": True,"use_gpu": True, "cuda_version": "12.6" 
            },
    target_platforms=["SNOWPARK_CONTAINER_SERVICES"]
)


### Verify Model Registration

In [ ]:
# Usage Example
from snowflake.ml.registry import Registry
reg = Registry(session=session) 
model_ref = reg.show_models()
model_ref

In [ ]:
# mv is a snowflake.ml.model.ModelVersion object
mv.create_service(service_name="DEFECTDETECTSERVICE",
                  service_compute_pool="PCB_CV_SERVICE_COMPUTEPOOL",
                  ingress_enabled=False,
                  image_repo="PCB_CV.PUBLIC.IMAGE_REPO",
                  max_instances=1,
                  gpu_requests=1)

## 6. Run Inference

Detect defects on test images using the deployed model service.

**Process:**
1. Get model reference from registry
2. Load test image as base64
3. Call inference service
4. Visualize results

In [ ]:

m = reg.get_model("DEFECTDETECTIONMODEL")
mv = m.version("v1")


df=session.table("TEST_DATA").limit(1).to_pandas()

first_row = df.iloc[0]
base64_image = first_row['IMAGE_DATA'] 
image_data_df = pd.DataFrame({'IMAGE_DATA': [base64_image]})


remote_prediction = mv.run(image_data_df, function_name="predict", service_name="DEFECTDETECTSERVICE")
remote_prediction

### Visualize Detection Results

In [ ]:
import json
import base64
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import io

# Class mapping dictionary
classes_la = {
    0: "background",
    1: "open",
    2: "short",
    3: "mousebite",
    4: "spur",
    5: "copper",
    6: "pin-hole"
}

# Function to display the image with bounding boxes and class labels
def display_image_with_boxes(image, boxes, labels, scores, target_size=(800, 600)):
    # Resize the image to a target size
    img = image.resize(target_size).convert("RGB")  # Resize and convert to RGB
    img_np = np.array(img)

    # Adjust the DPI and figure size
    fig, ax = plt.subplots(figsize=(3, 6), dpi=10)  # Adjust figure size and DPI
    ax.imshow(img_np)

    for label, box, score in zip(labels, boxes, scores):
        xmin, ymin, xmax, ymax = box
        class_label = classes_la[label]

        # Create a Rectangle patch
        rect = patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, linewidth=2, edgecolor='r', facecolor='none')
        ax.text(xmin, ymin, f"{class_label}: {score:.2f}", verticalalignment='top', color='red', fontsize=13, weight='bold')
        ax.add_patch(rect)

    plt.axis('off')
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)  # Ensure no padding/margins around the image
    plt.show()

# Combine the image data and remote prediction DataFrames
combined_df = pd.concat([image_data_df, remote_prediction], axis=1)

# Create a list to store data for the final DataFrame
rows = []

# Iterate through each row in the combined DataFrame
for index, row in combined_df.iterrows():
    output_str = row.get('output', None)  # Get the output column value

    if isinstance(output_str, str):  # Ensure it's a valid string before loading as JSON
        try:
            # Convert the 'output' column JSON string into a dictionary
            output_data = json.loads(output_str)

            # Extract boxes, labels, and scores from JSON data
            if 'boxes' in output_data and 'labels' in output_data and 'scores' in output_data:
                boxes = output_data['boxes']
                labels = output_data['labels']
                scores = output_data['scores']

                # Decode the image data
                image_data = base64.b64decode(row['IMAGE_DATA'])
                image = Image.open(io.BytesIO(image_data)).convert("RGB")

                # Limit to top 5 classes based on scores
                if len(scores) > 0:
                    # Create a DataFrame to manage boxes, labels, and scores
                    data = pd.DataFrame({
                        'box': boxes,
                        'label': labels,
                        'score': scores
                    })

                    # Get the top 5 entries based on scores
                    top_classes = data.nlargest(3, 'score')

                    # Extract corresponding boxes, labels, and scores
                    top_boxes = top_classes['box'].tolist()
                    top_labels = top_classes['label'].tolist()
                    top_scores = top_classes['score'].tolist()

                    # Store each of the top 5 predictions as a separate row
                    for i in range(len(top_boxes)):
                        rows.append({
                            'image_data': row['IMAGE_DATA'],
                            'output': row['output'],
                            'label': top_labels[i],
                            'box': top_boxes[i],
                            'score': top_scores[i]
                        })

                    # Display the image with bounding boxes and labels
                    display_image_with_boxes(image, top_boxes, top_labels, top_scores)
                else:
                    print("No scores available to limit to top 5.")
            else:
                print("Missing keys 'boxes', 'labels', or 'scores' in the output data.")

        except json.JSONDecodeError:
            print(f"Invalid JSON in row {index}, skipping this row.")
    else:
        print(f"Invalid output type (not a string) in row {index}, skipping this row.")

# Create the final DataFrame with the collected rows (one row per label/box/score)
final_df = pd.DataFrame(rows)
session.sql("create TABLE if not exists PCB_CV.PUBLIC.DETECTION_OUTPUTS (\
	image_data VARCHAR(16777216),\
	output VARCHAR(16777216),\
	label NUMBER(38,0),\
	box VARIANT,\
	score FLOAT\
)").collect()

# Write the DataFrame to the Snowflake table
combined_spdf = session.create_dataframe(final_df)
combined_spdf.write.save_as_table("DETECTION_OUTPUTS", mode="overwrite")
